In [1]:
from singleCAM_IROS._pipeline_support import _handle_dirpaths
from typing import Callable

from pathlib import Path

from numpy.typing import NDArray
import numpy as np
from astropy.io.fits.fitsrec import FITS_rec

from bloodmoon.io import simulation_files
from bloodmoon.mask import CodedMaskCamera, codedmask, count
from bloodmoon.images import argmax

import darksun as ds

ds.show.set_figures_darkbkg()

In [11]:
MASK_FITS: str = "mask_050_1040x17_20260129_CORRECTED.fits"
# MASK_FITS: str = "wfm_mask_NTHT_20250725.fits"

# SKYFIELD: str = "IROSDummy"
SKYFIELD: str = "Crab"

# DATA_FITS: str = "crab_mask_050_1040x17_20260206_2-50keV_1ks"
# DATA_FITS: str = "crab_mask_050_1040x17_20260206_2-50keV_1ks"
DATA_FITS: str = "crab_30deg_100s"


ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"

UPS_X: int = 5
UPS_Y: int = 1

In [12]:
# load filepaths
mask_path, simul_data, save_path = _handle_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
)
wfm: CodedMaskCamera = codedmask(mask_path, UPS_X, UPS_Y)
filepaths: dict[str, dict[str, Path]] = simulation_files(simul_data)


# data from camera A
catalogueA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlA_detct = ds.get_data(filepaths[ID_CAMERA_A]['detected'])
sdlA_recnstr = ds.get_data(filepaths[ID_CAMERA_A]['reconstructed'])

## data from camera B
catalogueB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])
sdlB_detct = ds.get_data(filepaths[ID_CAMERA_B]['detected'])
sdlB_recnstr = ds.get_data(filepaths[ID_CAMERA_B]['reconstructed'])

In [13]:
def get_phs_IDs(data: FITS_rec) -> NDArray:
    """Extracts the photons IDs and status flags from given phs record."""
    return data['ID']


detected_IDs_camA: NDArray = get_phs_IDs(sdlA_detct.DLdata)
reconsrt_IDs_camA: NDArray = get_phs_IDs(sdlA_recnstr.DLdata)

detected_IDs_camB: NDArray = get_phs_IDs(sdlB_detct.DLdata)
reconsrt_IDs_camB: NDArray = get_phs_IDs(sdlB_recnstr.DLdata)


unqIDs_dtc_camA, unqIDs_rct_camA = map(lambda x: len(np.unique(x)), (detected_IDs_camA, reconsrt_IDs_camA))
unqIDs_dtc_camB, unqIDs_rct_camB = map(lambda x: len(np.unique(x)), (detected_IDs_camB, reconsrt_IDs_camB))

print(
    f'## Camera: {ID_CAMERA_A.upper()}\n'
    f'  * IDs in detected list: {len(detected_IDs_camA)}\n'
    f'  * Unique IDs in detected list: {unqIDs_dtc_camA}\n'
    f'  * Delta (detc - unique detc): {len(detected_IDs_camA) - unqIDs_dtc_camA}\n\n'

    f'  * Unique IDs in reconstr list: {unqIDs_rct_camA}\n'
    f'  * Delta (detc - rectr): {unqIDs_dtc_camA - unqIDs_rct_camA}\n\n'


    f'## Camera: {ID_CAMERA_B.upper()}\n'
    f'  * IDs in detected list: {len(detected_IDs_camB)}\n'
    f'  * Unique IDs in detected list: {unqIDs_dtc_camB}\n'
    f'  * Delta (detc - unique detc): {len(detected_IDs_camB) - unqIDs_dtc_camB}\n\n'

    f'  * Unique IDs in reconstr list: {unqIDs_rct_camB}\n'
    f'  * Delta (detc - rectr): {unqIDs_dtc_camB - unqIDs_rct_camB}\n\n'
)

## Camera: CAM1A
  * IDs in detected list: 5437
  * Unique IDs in detected list: 5437
  * Delta (detc - unique detc): 0

  * Unique IDs in reconstr list: 5431
  * Delta (detc - rectr): 6

## Camera: CAM1B
  * IDs in detected list: 5441
  * Unique IDs in detected list: 5441
  * Delta (detc - unique detc): 0

  * Unique IDs in reconstr list: 5440
  * Delta (detc - rectr): 1




In [14]:
from astropy.io.fits.fitsrec import FITS_rec

def organise_phs_list(data: FITS_rec, in_place: bool = True) -> FITS_rec:
    """Sorts **in-place** the given photons list by ID."""
        # in-place sort to avoid doubling memory usage with a copy
    if in_place:
        data.sort(order='ID')
    else:
        data = np.sort(data, order='ID')
    return data


def check_phs_list_degeneracy(ids: NDArray, verbose: bool = True) -> NDArray:
    """Checks the given **ID sorted** list of photons for repeating entries."""
    n_entries = len(ids)

    # scan sorted rec for ID groups *idxs* (order O(N))
    # - find starting idxs of all unique ID vals
    # - reintroduce idx 0 and shift idx vals by 1
    # - find repeating IDs counts (append `n_entries` for end grp)
    rpt_ids = (ids[1:] != ids[:-1])
    grp_starts = np.concatenate(
        ([0], np.flatnonzero(rpt_ids) + 1)
    )
    grp_cts = np.diff(
        np.concatenate((grp_starts, [n_entries]))
    )
    # filter for repeating vals
    is_duplicate = (grp_cts > 1)
    rpt_cts = grp_cts[is_duplicate]

    # check for no duplicates
    if not len(rpt_cts):
        if verbose:
            print('No repeating entries in given record.')
        return np.array([])
    
    rpt_starts = grp_starts[is_duplicate]
    rpt_vals = ids[rpt_starts]

    if verbose:
        n_unique = len(grp_cts)
        print(
            f'Unique entries: {n_unique} / {n_entries} ({n_unique * 100 / n_entries:.2f} %)\n'
            f'Repeating entries: {len(rpt_vals)}'
        )
    
    # define struct array for repeating IDs for optimised output
    cols = [('value', ids.dtype), ('rpts', np.int64), ('start_idx', np.int64)]
    out = np.empty(len(rpt_vals), dtype=cols)

    out['value'] = rpt_vals
    out['rpts'] = rpt_cts
    out['start_idx'] = rpt_starts

    return out


def get_repeating_phIDs(
    data: FITS_rec,
    rpt_vals: NDArray,
    idxs: int | slice,
) -> NDArray | tuple[NDArray, ...]:
    """Extracts entries from given array with repeating phs data."""
    if isinstance(idxs, (int, np.integer)):
        row = rpt_vals[idxs]
        return data[row['start_idx'] : row['start_idx'] + row['rpts']]
    
    selected_rpts = rpt_vals[idxs]
    rpts_entries = tuple(
        data[row['start_idx'] : row['start_idx'] + row['rpts']]
        for row in selected_rpts
    )

    return rpts_entries

In [15]:
sorted_phs_list: FITS_rec = organise_phs_list(sdlA_detct.DLdata)
rpting_phs = check_phs_list_degeneracy(sorted_phs_list['ID'])

No repeating entries in given record.


In [16]:
idxs: int | slice = slice(0, 100)
rpt_data = get_repeating_phIDs(sdlA_detct.DLdata, rpting_phs, idxs)
rpt_rec = np.concatenate(rpt_data)

ValueError: need at least one array to concatenate

In [ ]:
from bisect import bisect
from darksun.types import Tag

def get_px_idxs(
    camera: CodedMaskCamera,
    shift_x: float,
    shift_y: float,
) -> tuple[int, int]:
    """Convert phs coords to nearest discrete pixel indices."""
    bins = camera.bins_detector
    pos = (bisect(bins.y, shift_y) - 1, bisect(bins.x, shift_x) - 1)
    return pos

def distr_phs(camera: CodedMaskCamera, data: FITS_rec) -> NDArray:
    """Distributes given photons in 2D hist."""
    bins = camera.bins_detector
    counts, *_ = np.histogram2d(data["Y"], data["X"], bins=[bins.y, bins.x])
    return counts



hist = distr_phs(wfm, rpt_rec)

rpt_phs_tags: tuple[Tag, ...] = tuple(
    Tag('', *get_px_idxs(wfm, sx, sy)) for idx, sy, sx in zip(rpt_rec['ID'], rpt_rec['Y'], rpt_rec['X'])
)

_labels: dict = {
    'xlabel': 'fine dir [px]',
    'ylabel': 'coarse dir [px]',
    'cbarlabel': 'counts [ph]',
    'tags': rpt_phs_tags,
}
_imgkwargs: dict = {
    'aspect': hist.shape[1] / hist.shape[0],
    'vmin': 0.0,
    'cmap': 'gist_heat',
}
ds.image_plot(
    ds.map4image(
        img=hist + 0.1 * hist.max() * wfm.bulk,
        title='Duplicate Phs Hist',
        img_kwargs=_imgkwargs,
        **_labels,
    ),
)

NameError: name 'rpt_rec' is not defined

In [ ]:
sorted_phs_list_camB: FITS_rec = organise_phs_list(sdlB_detct.DLdata)
rpting_phs_camB = check_phs_list_degeneracy(sorted_phs_list_camB['ID'])

No repeating entries in given record.


In [ ]:
idxs: int | slice = slice(0, 100)
rpt_data_camB = get_repeating_phIDs(sdlB_detct.DLdata, rpting_phs_camB, idxs)
rpt_rec_camB = np.concatenate(rpt_data_camB)

In [ ]:
assert False

from numpy.typing import NDArray
import numpy as np

a: NDArray = np.random.randint(0, 75, 100) * 20
a: NDArray = np.arange(120)
a.sort()

a_rpt = check_phs_list_degeneracy(a)
print(a_rpt)

get_repeating_phIDs(a, a_rpt, slice(2, 5))

AssertionError: 

In [ ]:
assert False

from typing import NamedTuple

class Duplicate(NamedTuple):
    """
    Identifier for duplicated entries in record.

    Attributes:
        value (int): Entry value.
        rpts (int): Number of entry repetitions.
        idxs (slice): Value idxs in array.
    """
    value: int
    rpts: int
    idxs: slice


def organise_phs_list(record: FITS_rec) -> FITS_rec:
    """Sorts the given photons list by ID."""
    return np.sort(record, order='ID')

def check_phs_list_degeneracy(record: FITS_rec, verbose: bool = True) -> tuple[Duplicate, ...]:
    """Checks the given **ID sorted** list of photons for repeating entries."""
    unq, idxs, cts = np.unique(record['ID'], return_index=True, return_counts=True)
    rpts_vals = np.where((cts > 1))[0]
    if verbose:
        print(
            f'Unique record entries: {len(unq)} / {len(record)} ({len(unq) * 100 / len(record):.2f} %)\n'
            f'Repeating record entries: {len(rpts_vals)}\n'
        )
    duplicates = tuple(
        Duplicate(unq[rpts], cts[rpts], slice(idxs[rpts], idxs[rpts] + cts[rpts]))
        for rpts in rpts_vals
    )
    return duplicates


sorted_phs_list: FITS_rec = organise_phs_list(sdlA_detct.DLdata)
duplicates: tuple[Duplicate, ...] = check_phs_list_degeneracy(sorted_phs_list)

idx: int = 0
sorted_phs_list[duplicates[idx].idxs]